In [ ]:
import os
import glob

import numpy as np

import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

import sys

sys.path.append(os.path.abspath('..'))

from mrileyowens.dja import cutout, check

In [ ]:
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

def coords():

    '''
    Plot the F090W dropouts against the full footprint of the fields
    '''

    # Get the file paths to the full photometric catalogs
    files = glob.glob(f'{data}/catalogs/photometryCatalog_*_feb2026.fits')

    # For each catalog file
    for file in files:

        # Determine the string identifier for the field
        field = os.path.basename(file).split('_')[1]

        # Get the associated F090W dropout catalogs
        files_dropouts = glob.glob(f'{results}/catalogs/{field}_f090w_dropouts_aper_*.fits')

        # ------------------------------
        # Plot the catalog's full sample
        # ------------------------------

        # Get the catalog's HDU list
        hdul = fits.open(file)

        # Create a new figure to show the dropouts in the field against the full set of targets, for each aperture choice
        fig, ax = plt.subplots(len(files_dropouts), sharex=True, sharey=True, figsize=(5,5*len(files_dropouts)))

        # Get the coordinates of the whole catalog
        ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

        # Plot the whole-catalog coordinates on each panel
        for i in range(len(files_dropouts)): ax[i].scatter(ra, dec, c='black', marker='o', s=1)

        # For each F090W dropout catalog file
        for i, file in enumerate(files_dropouts):

            # Open the catalog's HDU list
            hdul = fits.open(file)

            # Get the coordinates of the dropouts
            ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

            # Plot the coordinates on the matching panel
            ax[i].scatter(ra, dec, c='red', marker='s', s=1, label='F090W dropouts')

            # Invert the x axis (right ascension) to matcht the standard direction of increase
            ax[i].xaxis.set_inverted(True)

            # Add a legend to the figure
            ax[i].legend(loc='upper right')

            # Add an annotation indicating the field
            at = AnchoredText(field.upper(), loc='upper left', frameon=False, prop=dict(fontweight='bold'))
            ax[i].add_artist(at)

        # Label the axes
        ax[-1].set_xlabel('Right ascension (deg.)')
        ax[1].set_ylabel('Declination (deg.)')

        # Save the figure
        fig.savefig(f'{figs}/{field}_f090w_dropouts.png', bbox_inches='tight', dpi=200)

        plt.close('all')

def footprints():

    #files = glob.glob(f'{results}/catalogs/*_f090w_dropouts_aper_0.fits')
    files = glob.glob(f'{data}/catalogs/photometryCatalog_*_feb2026.fits')

    filters = ['f090w','f115w','f150w','f200w','f277w','f356w','f410m','f444w']

    for file in files:

        hdul = fits.open(file)

        # Get the filters in the catalog
        #filters = [col.split('_')[0] for col in hdul[1].columns.names if col.endswith('_tot_0')]

        ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

        print(ra, dec)

        for filter in filters:

            #ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

            mask = ~np.isnan(hdul[1].data[f'{filter}_tot_0'])

            fig, ax = plt.subplots()

            ax.scatter(ra[mask], dec[mask], c='black', marker='o')

            ax.set_xlabel('Right ascension (deg.)')
            ax.set_ylabel('Declination (deg.)')

            ax.xaxis.set_inverted(True)

            at = AnchoredText(filter, loc='upper right')
            ax.add_artist(at)

            #fig.savefig(f'{figs}/{os.path.basename(file)}.png', bbox_inches='tight', dpi=200)

def cutouts():

    files = glob.glob(f'{results}/catalogs/*_f070w_dropouts_init.fits')

    for file in files:

        hdul = fits.open(file)

        ids = hdul[1].data['ID']
        ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

        for i, _ in enumerate(ra):

            cutout(ra[i], dec[i], filters=['f435w', 'f606w','f070w','f814w','f090w-clear','f150w-clear'], id=ids[i], save=True, dir='figs/f070w_dropouts_init_cutouts')
            #cutout(ra[i], dec[i], id=ids[i], save=True, dir='figs/f070w_dropouts_init_cutouts')

def check_dja():

    '''
    Check for extant spectroscopic observations of the F090W dropouts in the DJA catalog
    '''

    # Get the F090W dropout catalogs
    files = glob.glob(f'{results}/catalogs/*_f090w_dropouts_aper_*.fits')

    # For each catalog file
    for file in files:

        # Open the HDU list of the catalog
        hdul = fits.open(file)

        # Get the coordinates of the sources as a SkyCoord object
        coords = SkyCoord(ra=hdul[1].data['RA'] * u.deg, dec=hdul[1].data['DEC'] * u.deg)

        # Check for coordinate matches in the spectroscopic DJA catalog
        check(coords)